In [12]:
from urllib.request import urlopen
import shutil
import sys
from pathlib import Path



PROJECT_ROOT = Path("/Users/alessandro/Progetti/Ensembling")
DATA_DIR = PROJECT_ROOT / "data" / "nyt10m"
DATA_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = (
    "https://thunlp.oss-cn-qingdao.aliyuncs.com/"
    "opennre/benchmark/nyt10m"
)

filenames = [
    "nyt10m_rel2id.json",
    "nyt10m_train.txt",
    "nyt10m_val.txt",
    "nyt10m_test.txt",
]

for filename in filenames:
    destination = DATA_DIR / filename

    if destination.exists() and destination.stat().st_size > 0:
        print(f"Già presente: {filename}", flush=True)
        continue

    print(f"Scaricamento: {filename}...", flush=True)
    temporary = destination.with_suffix(destination.suffix + ".part")

    with urlopen(f"{BASE_URL}/{filename}", timeout=120) as response:
        expected_size = response.headers.get("Content-Length")

        with temporary.open("wb") as output:
            shutil.copyfileobj(response, output)

    if expected_size is not None:
        assert temporary.stat().st_size == int(expected_size), (
            f"Download incompleto: {filename}. Riesegui la cella."
        )

    temporary.replace(destination)
    print(f"Completato: {destination.stat().st_size / 1_000_000:.1f} MB")

print("Cartella dataset:", DATA_DIR)

Già presente: nyt10m_rel2id.json
Già presente: nyt10m_train.txt
Già presente: nyt10m_val.txt
Già presente: nyt10m_test.txt
Cartella dataset: /Users/alessandro/Progetti/Ensembling/data/nyt10m


In [13]:
import ast
import json
from pprint import pprint

def read_records(path):
    with path.open(encoding="utf-8") as file:
        for line in file:
            if not line.strip():
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                record = ast.literal_eval(line)

            yield record

with (DATA_DIR / "nyt10m_rel2id.json").open(encoding="utf-8") as file:
    rel2id = json.load(file)

print("Labels'number:", len(rel2id))
print("NA's ID:", rel2id["NA"])

example = next(read_records(DATA_DIR / "nyt10m_train.txt"))
pprint(example)

print("Phrase:", example["text"])
print("Entity head:", example["h"]["name"])
print("Entity tail:", example["t"]["name"])
print("Relation:", example["relation"])

for split in ["train", "val", "test"]:
    path = DATA_DIR / f"nyt10m_{split}.txt"
    count = sum(1 for _ in read_records(path))
    print(f"{split}: {count:,} record")

Labels'number: 25
NA's ID: 0
{'h': {'id': 'm.059rby', 'name': 'New York', 'pos': [33, 41]},
 'relation': '/location/location/contains',
 't': {'id': 'm.016p8t', 'name': 'Carnegie Hall', 'pos': [73, 86]},
 'text': 'One of the newest auditoriums in New York , built into the bedrock '
         'below Carnegie Hall , has added a dose of richness to the concert '
         'scene .'}
Phrase: One of the newest auditoriums in New York , built into the bedrock below Carnegie Hall , has added a dose of richness to the concert scene .
Entity head: New York
Entity tail: Carnegie Hall
Relation: /location/location/contains
train: 417,893 record
val: 46,422 record
test: 11,086 record
